# N-граммная языковая модель

Построим модель, которая продолжает обращение по частотам соседних слов. Сначала проверим её на трёх предложениях из лекции, затем сравним униграммы, биграммы и триграммы на корпусе «Север». В конце прочитаем несколько продолжений и попробуем объяснить, почему они получились именно такими.

In [ ]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Подготовка последовательностей

Берём по одному обращению из каждой группы. Токенизатор выделяет слова и числа; пунктуация в этой работе не предсказывается. `BOS` задаёт начало истории, `EOS` обозначает предсказываемое завершение, `UNK` заменяет слова вне словаря.

Функция `events` уже написана. Она выдаёт пары «история, следующий токен» и начинает историю заново для каждого предложения. Посмотрите на пример её работы перед тем, как писать счётчики.

In [ ]:
from collections import Counter, defaultdict
import math

BOS, EOS, UNK = "<BOS>", "<EOS>", "<UNK>"
def tokenize(text):
    return re.findall(r"[а-яёa-z0-9]+", text.lower())

df = pd.read_csv(DATA / "requests.csv", keep_default_na=False).drop_duplicates("group_id")
parts = {name: [tokenize(t) for t in df.loc[df["split"].eq(name), "text"]]
         for name in ["train", "validation", "test"]}
assert tokenize("Студент читает книгу") == ["студент", "читает", "книгу"]
print({name: len(values) for name, values in parts.items()})

# Готовый итератор отделяет устройство истории от подсчёта частот.
def events(sentence, n):
    sequence = [BOS] * (n - 1) + list(sentence) + [EOS]
    for t in range(n - 1, len(sequence)):
        history = tuple(sequence[t - n + 1:t]) if n > 1 else ()
        yield history, sequence[t]


In [ ]:
demo_sentence = tokenize("студент читает книгу")
for history, target in events(demo_sentence, 2):
    print(history, "→", target)

У биграммы в истории один токен. Последняя цель — EOS, а первая история — BOS. Для триграммы перед первым словом появились бы два BOS, но число целей осталось бы тем же.

## 1. Общий словарь

Посчитайте частоты слов только в `parts["train"]`. Сохраните множество слов с частотой не меньше двух в `known`. Создайте отсортированный список `vocab`, добавив `EOS` и `UNK` и исключив `BOS`.

Создайте `encoded`: словарь с ключами train, validation, test и списками токенизированных предложений. Во всех частях заменяйте слова вне `known` на `UNK`. Посчитайте долю UNK среди обычных слов каждой части, без EOS. Что эта доля говорит о сохранённых различиях текста?

In [ ]:
# Ваш код
raise NotImplementedError

In [ ]:
assert BOS not in vocab and EOS in vocab and UNK in vocab
assert all(t in vocab for sentences in encoded.values() for sentence in sentences for t in sentence)
assert len(encoded["train"]) == 72 and len(encoded["validation"]) == len(encoded["test"]) == 24

## 2. Счётчики переходов

Напишите `fit_model(sentences, n, vocab)` с помощью `events`. Для каждой истории считайте отдельные продолжения. Верните словарь с полями `n`, `vocab` (tuple), `counts` (история → Counter токенов), `totals` (история → общее число продолжений).

Проверьте три предложения из лекции. После «студент» должны получиться два «читает» и одно «возвращает». Объясните, почему всего целей двенадцать, хотя обычных слов девять.

In [ ]:
def fit_model(sentences, n, vocab):
    # Ваш код
    raise NotImplementedError


In [ ]:
toy_sentences = [tokenize(t) for t in ["студент читает книгу", "студент читает журнал", "студент возвращает книгу"]]
toy_vocab = sorted({t for s in toy_sentences for t in s} | {EOS, UNK})
toy_model = fit_model(toy_sentences, 2, toy_vocab)
assert len(toy_vocab) == 7
assert toy_model["counts"][("студент",)]["читает"] == 2
assert toy_model["totals"][("студент",)] == 3
assert sum(toy_model["totals"].values()) == 12  # 9 слов + 3EOS
assert all(BOS not in continuation for continuation in toy_model["counts"].values())

## 3. Вероятность нового продолжения

Реализуйте `probability(model, history, target, alpha=0.1)` по add-alpha. При `alpha <= 0` или цели вне словаря вызывайте `ValueError`. Для невстречавшейся истории счётчики равны нулю.

Готовая `distribution` собирает значения в порядке `model["vocab"]`. Примените её к истории «студент» при alpha=1 и сверьте семь вероятностей с лекцией. Затем возьмите невстречавшуюся историю. Почему у неё получается равномерное распределение?

In [ ]:
def probability(model, history, target, alpha=0.1):
    # Ваш код
    raise NotImplementedError

def distribution(model, history, alpha=0.1):
    return np.array([probability(model, history, t, alpha) for t in model["vocab"]])


In [ ]:
assert np.isclose(probability(toy_model, ("студент",), "читает", 1), 0.3)
assert np.isclose(probability(toy_model, ("студент",), "возвращает", 1), 0.2)
for h in [(BOS,), ("студент",), ("невстречавшаяся_история",)]:
    p = distribution(toy_model, h, 1)
    assert (p > 0).all() and np.isclose(p.sum(), 1)
assert np.allclose(distribution(toy_model, ("невстречавшаяся_история",), 1), np.ones(7)/7)

## 4. Perplexity и выбор порядка

Напишите `evaluate_ppl(model, sentences, alpha=0.1)`. Суммируйте −ln вероятностей целей, включая EOS. Верните словарь `ppl`, `mean_nll`, `targets`. При отсутствии целевых событий вызывайте `ValueError`; одно пустое предложение всё ещё содержит цель EOS.

Создайте `models` для n=1,2,3 на одном `encoded["train"]`. Соберите DataFrame `comparison` с колонками `n`, `train_ppl`, `validation_ppl`, `train_targets`, `validation_targets`. Выберите `selected_n` по наименьшей validation perplexity, при равенстве — меньший n. Сохраните модель в `selected`, её тестовую оценку — в `test_result`.

Словарь и alpha остаются общими. Почему число целей в одной части должно совпадать у всех трёх порядков?

In [ ]:
def evaluate_ppl(model, sentences, alpha=0.1):
    # Ваш код
    raise NotImplementedError

# Ваш код
raise NotImplementedError

In [ ]:
assert comparison["train_targets"].nunique() == comparison["validation_targets"].nunique() == 1
assert test_result["targets"] == sum(len(s)+1 for s in encoded["test"])
assert math.isfinite(test_result["ppl"]) and test_result["ppl"] >= 1
assert evaluate_ppl(selected, [[]])["targets"] == 1
assert np.isclose(math.exp(-np.log([0.5, 0.25, 0.5]).mean()), 16**(1/3))

## 5. Продолжаем префикс

Закончите `generate(model, prefix="как", mode="sample", seed=42, max_new=40)`. Преобразуйте префикс тем же словарём. На каждом шаге получите распределение по последним n−1 токенам; для униграммы история пустая. В greedy выбирайте максимум, в sample используйте `rng.choice` с вероятностями.

При EOS завершайте генерацию, иначе продолжайте до `max_new`. Верните `text` (префикс и продолжение), `tokens` (только новые токены без EOS), `stopped_by` ("EOS" или "limit") и `steps` (число новых предсказаний, включая EOS, если он появился). Оставляйте UNK видимым.

Соберите `generations`: один результат greedy и sampling с seed 7, 42 и 99. Сравните повтор с одним seed и результаты разных seed. Чем условия этого опыта отличаются от расчёта perplexity?

In [ ]:
def generate(model, prefix="как", mode="sample", seed=42, max_new=40):
    # Ваш код
    raise NotImplementedError


In [ ]:
assert generate(selected, seed=42) == generate(selected, seed=42)
for result in [generate(selected, mode="greedy"), generate(selected, seed=7)]:
    assert 1 <= result["steps"] <= 40
    assert result["stopped_by"] in {"EOS", "limit"}
    assert all(t in vocab and t not in {BOS, EOS} for t in result["tokens"])
    if result["stopped_by"] == "limit": assert len(result["tokens"]) == 40

## Состояние после нулевого входа

Следующая ячейка воспроизводит скалярную RNN из лекции. Параметры заданы вручную. Сначала предскажите знак второго состояния, затем сравните цепочку входов (1;0) с (0;0).

In [ ]:
# Скалярная RNN из лекции: параметры заданы вручную, обучение не выполняется.
h = 0.0
for x in [1.0, 0.0]:
    h = np.tanh(0.5*h + x)
    print("input", x, "state", h)

## Разбор результатов

Объясните выбор порядка по полученной таблице. Сопоставьте perplexity с долей UNK: какие различия исходных слов модель ещё способна учитывать? Затем сравните два продолжения, указав, как они завершились и какие переходы выглядят странно.

Для примера RNN объясните, почему при втором нулевом входе состояние осталось ненулевым. Сдайте выполненный ноутбук с этими пояснениями; перед сдачей перезапустите ядро и выполните все ячейки по порядку.